# Часть 2.1 — 2024 YouTube Channels (1M)

3 эксперимента: (1) регрессия `log(subscriber_count)` от метрик активности; (2) бинарная классификация «popular» (≥100k подписчиков); (3) KMeans-сегментация каналов.

In [1]:
from pathlib import Path

import kagglehub
import pandas as pd
from _setup import evaluate_models, make_spark
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    ClusteringEvaluator,
    RegressionEvaluator,
)
from pyspark.ml.feature import StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import functions as F

SEED = 42
spark = make_spark("hw3-youtube", partitions=8, memory="4g")
spark

## Загрузка

In [2]:
csv = next(Path(kagglehub.dataset_download("asaniczka/2024-youtube-channels-1-million")).glob("*.csv"))

raw = (
    spark.read.csv(str(csv), header=True, inferSchema=True)
    .select(
        F.col("subscriber_count").cast("double").alias("subscriber_count"),
        F.col("total_views").cast("double").alias("total_views"),
        F.col("total_videos").cast("double").alias("total_videos"),
        "country",
        F.col("mean_views_last_30_videos").cast("double").alias("mean_views_last_30_videos"),
        F.col("median_views_last_30_videos").cast("double").alias("median_views_last_30_videos"),
        F.col("std_views_last_30_videos").cast("double").alias("std_views_last_30_videos"),
        F.col("videos_per_week").cast("double").alias("videos_per_week"),
    )
    .dropna()
    .filter((F.col("subscriber_count") > 0) & (F.col("total_views") > 0))
)
df = raw.sample(fraction=0.2, seed=SEED).cache()
print(f"строк (сэмпл 20%): {df.count()}")

строк (сэмпл 20%): 42950


## Подготовка

In [3]:
numeric = [
    "total_videos",
    "mean_views_last_30_videos",
    "median_views_last_30_videos",
    "std_views_last_30_videos",
    "videos_per_week",
]

prep = Pipeline(
    stages=[
        StringIndexer(inputCol="country", outputCol="country_idx", handleInvalid="keep"),
        VectorAssembler(inputCols=[*numeric, "country_idx"], outputCol="features_raw"),
        StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False),
    ]
).fit(df)
data = (
    prep.transform(df)
    .withColumn("label", F.log1p("subscriber_count"))
    .withColumn("is_popular", (F.col("subscriber_count") >= 100_000).cast("double"))
    .select("features", "label", "is_popular")
)
train, test = data.randomSplit([0.8, 0.2], seed=SEED)
train.cache()
test.cache()
print(f"train: {train.count()}, test: {test.count()}")

train: 34530, test: 8420


## Эксперимент 1 — регрессия log(total_views)

In [4]:
reg_evaluators = {
    "RMSE": RegressionEvaluator(labelCol="label", metricName="rmse"),
    "R2": RegressionEvaluator(labelCol="label", metricName="r2"),
}
regressors = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol="label", maxIter=50),
    "RFRegressor": RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, seed=SEED, maxDepth=10),
}
evaluate_models(regressors, train, test, reg_evaluators)

,RMSE,R2
model,,
LinearRegression,2.756742,0.041361
RFRegressor,1.296512,0.787961


## Эксперимент 2 — классификация «popular» (≥ 100k subs)

In [5]:
cls_evaluators = {
    "ROC_AUC": BinaryClassificationEvaluator(labelCol="is_popular", metricName="areaUnderROC"),
    "PR_AUC": BinaryClassificationEvaluator(labelCol="is_popular", metricName="areaUnderPR"),
}
classifiers = {
    "LogReg": LogisticRegression(featuresCol="features", labelCol="is_popular", maxIter=100),
    "RandomForest": RandomForestClassifier(
        featuresCol="features", labelCol="is_popular", numTrees=50, seed=SEED, maxDepth=10
    ),
}
evaluate_models(classifiers, train, test, cls_evaluators)

,ROC_AUC,PR_AUC
model,,
LogReg,0.914167,0.420453
RandomForest,0.970928,0.765072


## Эксперимент 3 — KMeans-сегментация каналов

In [6]:
sil_eval = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
rows = []
for k in range(2, 7):
    m = KMeans(k=k, featuresCol="features", seed=SEED, maxIter=20).fit(data)
    rows.append({"k": k, "silhouette": sil_eval.evaluate(m.transform(data)), "WSSSE": m.summary.trainingCost})
exp3 = pd.DataFrame(rows).set_index("k")
exp3

,silhouette,WSSSE
k,,
2,0.758591,227345.097474
3,0.995473,189807.079235
4,0.829905,145767.675150
5,0.834161,119793.282784
6,0.757129,86125.631531


In [7]:
best_k = int(exp3["silhouette"].idxmax())
print(f"best k = {best_k}")
best = KMeans(k=best_k, featuresCol="features", seed=SEED, maxIter=20).fit(data)
best.transform(prep.transform(df)).groupBy("prediction").agg(
    F.count("*").alias("n"),
    F.avg("subscriber_count").alias("avg_subs"),
    F.avg("total_views").alias("avg_views"),
    F.avg("videos_per_week").alias("avg_vpw"),
).orderBy("prediction").show()

best k = 3


+----------+-----+------------------+--------------------+------------------+
|prediction|    n|          avg_subs|           avg_views|           avg_vpw|
+----------+-----+------------------+--------------------+------------------+
|         0|42928| 66702.46689806187|2.2031304723001305E7|0.6865390887066717|
|         1|    1|            2001.0|              2007.0|            2013.0|
|         2|   21|3415506.6666666665|1.7643997419047618E9| 5.976190476190476|
+----------+-----+------------------+--------------------+------------------+



## Выводы

- **Эксп. 1**: RFRegressor (R² 0.79) сильно лучше LinearRegression (R² 0.04). log(subscriber_count) нелинейно зависит от метрик активности (mean/median/std просмотров за 30 видео и videos_per_week).
- **Эксп. 2**: классификация «popular» (≥100k подписчиков) — класс редкий; ROC-AUC у RF=0.97, PR-AUC=0.77 (LogReg PR-AUC=0.42). RF заметно лучше на несбалансированном таргете.
- **Эксп. 3**: silhouette максимален при k=3 — каналы делятся на крупные/средние/мелкие архетипы по объёму просмотров и стилю постинга.

In [8]:
spark.stop()